# Gripper Dataset Quality Control & Exploration Notebook
**Project:** Computer Vision – Gripper Pose Estimation & Dataset Quality Assessment  
**Dataset:** `mv-umi-markers-placement` (LeRobot v3 schema)  
**Objective:** Interactive exploratory analysis of dataset structure, video streams, kinematics, and marker visibility.

In [ ]:
import os
import sys
import cv2
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, "../src")
from dataset_discovery import DatasetDiscovery
from image_quality import ImageQualityAuditor
from marker_visibility import MarkerVisibilityAuditor

## 1. Dataset Inventory & Metadata Discovery

In [ ]:
discovery = DatasetDiscovery("../../mv-umi-markers-placement", "../outputs")
inv = discovery.discover()
print(f"Total Demonstrations: {inv['total_episodes_in_primary_manifest']}")
print(f"Total Storage: {inv['total_storage_mb']} MB")
print(f"Format: {inv['format']}")

## 2. Load Episode Manifest & Kinematic Telemetry

In [ ]:
data_path = "../../mv-umi-markers-placement/data/chunk-000/file-000.parquet"
df_data = pq.read_table(data_path, columns=["episode_index", "frame_index", "timestamp", "observation.state"]).to_pandas()
states = np.array(df_data["observation.state"].tolist())
gripper_w_mm = states[:, 6] * 1000.0

plt.figure(figsize=(8, 4))
plt.hist(gripper_w_mm, bins=35, color="#1565c0", edgecolor="black", alpha=0.8)
plt.title("Gripper Opening Width Distribution")
plt.xlabel("Width (mm)")
plt.ylabel("Frame Count")
plt.show()

## 3. Visual Stream Inspection & Marker Visibility Test

In [ ]:
cap = cv2.VideoCapture("../../mv-umi-markers-placement/videos/observation.images.camera0_rgb/chunk-000/file-000.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, 500)
ret, frame = cap.read()
cap.release()

marker_auditor = MarkerVisibilityAuditor()
m_res = marker_auditor.evaluate_frame(frame, "camera0_rgb")
print("Marker Detection Result:", m_res["marker_visibility_status"])
print("Fiducials Detected:", m_res["markers_detected_count"])
print("Left Flap Coverage:", m_res["left_flap_coverage"])
print("Right Flap Coverage:", m_res["right_flap_coverage"])

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.imshow(frame_rgb)
plt.title(f"Camera0 Frame 500 - Status: {m_res['marker_visibility_status']}")
plt.axis("off")
plt.show()